# Data Description
- Background: 台北市 YouBike 2.0 原始交易資料，用於分析 NTU 區域的 YouBike 使用情況。資料包含每次租借交易的完整紀錄，包括借還車時間、站點、租借時長、車輛類型等資訊。
- date received: 202403-202501（持續更新）
- Path to data file: `data/raw/previews/YYYYMM_YouBike_preview.csv`（預覽資料），完整資料路徑：`data/raw/YYYYMM_YouBike.csv`
- Unit of observation: 單次 YouBike 租借交易紀錄
- Sample period: 2024年3月 - 2025年1月
- Known issues:
    - 原始資料檔案較大（每個月約200MB+），預覽檔僅為樣本
    - 部分站點名稱可能包含特殊字元或格式不一致
    - 租借時長為時間格式（HH:MM:SS），需轉換為數值才能進行統計分析
- Definition for each variable: 
    - borrow_datetime: 借車時間（日期時間格式，YYYY-MM-DD HH:MM:SS）
    - borrow_site: 借車站點名稱
    - return_datetime: 還車時間（日期時間格式，YYYY-MM-DD HH:MM:SS）
    - return_site: 還車站點名稱
    - borrowing_time: 租借時長（時間格式，HH:MM:SS）
    - type: 車輛類型（一般車/電輔車）
    - date: 交易日期（YYYY-MM-DD） 


In [ ]:
import pandas as pd
import glob
import os

## 資料設定

In [ ]:
# 選擇要分析的資料檔案（可選擇單一月份或合併所有月份）
# 單一月份範例（使用預覽檔）：
input_data_file = "./previews/202501_YouBike_preview.csv"

# 或合併所有月份預覽資料（取消註解以使用）：
# preview_files = glob.glob("./previews/*_preview.csv")
# preview_files.sort()
# input_data_file = preview_files  # 將讀取多個檔案

# 如需使用完整原始資料（檔案較大），可取消註解以下程式碼：
# input_data_file = "./202501_YouBike.csv"

## Summary

In [ ]:
# 讀取資料
if isinstance(input_data_file, list):
    # 如果 input_data_file 是列表，合併所有檔案
    df_list = []
    for file in input_data_file:
        df_temp = pd.read_csv(file)
        df_list.append(df_temp)
    df = pd.concat(df_list, ignore_index=True)
else:
    # 讀取單一檔案
    df = pd.read_csv(input_data_file)

print(f"資料形狀: {df.shape}")
print(f"欄位: {list(df.columns)}")
print(f"\n資料筆數: {len(df)}")

### Sample

In [4]:
df.head(10)

,sepal.length,sepal.width,petal.length,petal.width,variety
0,5.1,3.5,1.4,0.2,Setosa
1,4.9,3.0,1.4,0.2,Setosa
2,4.7,3.2,1.3,0.2,Setosa
3,4.6,3.1,1.5,0.2,Setosa
4,5.0,3.6,1.4,0.2,Setosa
5,5.4,3.9,1.7,0.4,Setosa
6,4.6,3.4,1.4,0.3,Setosa
7,5.0,3.4,1.5,0.2,Setosa
8,4.4,2.9,1.4,0.2,Setosa
9,4.9,3.1,1.5,0.1,Setosa


### Summary Stats

In [5]:
df.describe()

,sepal.length,sepal.width,petal.length,petal.width
count,150.000000,150.000000,150.000000,150.000000
mean,5.843333,3.057333,3.758000,1.199333
std,0.828066,0.435866,1.765298,0.762238
min,4.300000,2.000000,1.000000,0.100000
25%,5.100000,2.800000,1.600000,0.300000
50%,5.800000,3.000000,4.350000,1.300000
75%,6.400000,3.300000,5.100000,1.800000
max,7.900000,4.400000,6.900000,2.500000


In [ ]:
# 基本資料品質檢查
print("=" * 50)
print("資料品質檢查")
print("=" * 50)

# 檢查缺失值
print("\n各欄位缺失值數量:")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "無缺失值")

# 檢查資料類型
print("\n各欄位資料類型:")
print(df.dtypes)

# 檢查唯一值數量
print("\n各欄位唯一值數量:")
for col in df.columns:
    print(f"  {col}: {df[col].nunique()}")

# 檢查日期範圍
if 'date' in df.columns:
    df['date_parsed'] = pd.to_datetime(df['date'])
    print(f"\n日期範圍: {df['date_parsed'].min()} 至 {df['date_parsed'].max()}")

# 檢查車輛類型分布
if 'type' in df.columns:
    print(f"\n車輛類型分布:")
    print(df['type'].value_counts())

# 檢查時間邏輯：還車時間應該晚於借車時間
if 'borrow_datetime' in df.columns and 'return_datetime' in df.columns:
    df['borrow_dt'] = pd.to_datetime(df['borrow_datetime'])
    df['return_dt'] = pd.to_datetime(df['return_datetime'])
    invalid_time = (df['return_dt'] < df['borrow_dt']).sum()
    print(f"\n還車時間早於借車時間的記錄數: {invalid_time}")

# 檢查最常借車/還車的站點
if 'borrow_site' in df.columns:
    print(f"\n最常借車的前5個站點:")
    print(df['borrow_site'].value_counts().head())
    
if 'return_site' in df.columns:
    print(f"\n最常還車的前5個站點:")
    print(df['return_site'].value_counts().head())
